In [ ]:
!pip -q install rpy2
%load_ext rpy2.ipython

In [ ]:
%%R
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, ggplot2, dplyr, lubridate, stringr, readxl, data.table, gdata, scales)

# Define variable locations (v1996 forms)

In [ ]:
%%R
hcris.vars = NULL
hcris.vars = rbind(hcris.vars,c('beds','S300001','01200','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('tot_charges','G300000','00100','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('tot_discounts','G300000','00200','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('tot_operating_exp','G300000','00400','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('ip_charges','G200000','00100','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('icu_charges','G200000','01500','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('ancillary_charges','G200000','01700','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('tot_discharges','S300001','00100','1500','numeric'))
hcris.vars = rbind(hcris.vars,c('mcare_discharges','S300001','00100','1300','numeric'))
hcris.vars = rbind(hcris.vars,c('mcaid_discharges','S300001','00100','1400','numeric'))
hcris.vars = rbind(hcris.vars,c('tot_mcare_payment','E00A18A','01600','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('secondary_mcare_payment','E00A18A','01700','0100','numeric'))
hcris.vars = rbind(hcris.vars,c('street','S200000','00100','0100','alpha'))
hcris.vars = rbind(hcris.vars,c('city','S200000','00101','0100','alpha'))
hcris.vars = rbind(hcris.vars,c('state','S200000','00101','0200','alpha'))
hcris.vars = rbind(hcris.vars,c('zip','S200000','00101','0300','alpha'))
hcris.vars = rbind(hcris.vars,c('county','S200000','00101','0400','alpha'))
colnames(hcris.vars)=c("variable","WKSHT_CD","LINE_NUM","CLMN_NUM","source")

# Extract data from v1996 forms (2009-2011)

In [ ]:
%%R
for (i in 2009:2011) {
  HCRIS.alpha=read_csv(paste0("../../hcris-data/HCRIS_v1996/HospitalFY",i,"/hosp_",i,"_ALPHA.CSV"),
                       col_names=c('RPT_REC_NUM','WKSHT_CD','LINE_NUM','CLMN_NUM','ITM_VAL_NUM'))
  HCRIS.numeric=read_csv(paste0("../../hcris-data/HCRIS_v1996/HospitalFY",i,"/hosp_",i,"_NMRC.CSV"),
                         col_names=c('RPT_REC_NUM','WKSHT_CD','LINE_NUM','CLMN_NUM','ITM_VAL_NUM'))
  HCRIS.report=read_csv(paste0("../../hcris-data/HCRIS_v1996/HospitalFY",i,"/hosp_",i,"_RPT.CSV",sep=""),
                        col_names=c('RPT_REC_NUM','PRVDR_CTRL_TYPE_CD','PRVDR_NUM','NPI',
                                    'RPT_STUS_CD','FY_BGN_DT','FY_END_DT','PROC_DT',
                                    'INITL_RPT_SW','LAST_RPT_SW','TRNSMTL_NUM','FI_NUM',
                                    'ADR_VNDR_CD','FI_CREAT_DT','UTIL_CD','NPR_DT',
                                    'SPEC_IND','FI_RCPT_DT'))
  final.reports = HCRIS.report %>%
    select(report=RPT_REC_NUM, provider_number=PRVDR_NUM, npi=NPI, 
           fy_start=FY_BGN_DT, fy_end=FY_END_DT, date_processed=PROC_DT, 
           date_created=FI_CREAT_DT, status=RPT_STUS_CD) %>%
    mutate(year=i)
  
  for (v in 1:nrow(hcris.vars)) {
    hcris.data=get(paste("HCRIS.",hcris.vars[v,5],sep=""))
    var.name=quo_name(hcris.vars[v,1])    
    val = hcris.data %>%
      filter(WKSHT_CD==hcris.vars[v,2], LINE_NUM==hcris.vars[v,3], CLMN_NUM==hcris.vars[v,4]) %>%
      select(report=RPT_REC_NUM, !!var.name:=ITM_VAL_NUM) 
    assign(paste("val.",v,sep=""),val)
    final.reports=left_join(final.reports, 
              get(paste("val.",v,sep="")),
              by="report")
  }
  assign(paste("final.reports.",i,sep=""),final.reports)
  if (i==2009) {
    final.hcris.v1996=final.reports.2009
  } else {
    final.hcris.v1996=rbind(final.hcris.v1996,get(paste("final.reports.",i,sep="")))
  }
  
}

# Save output

In [ ]:
%%R
write_csv(final.hcris.v1996, '../data/output/HCRIS_Data_v1996.csv')